# MazeGPT: Reactive 2D Maze Navigation
### Google Colab Pure RL Execution Manual (Zero Pretraining)

> **Protocol**: Reactive navigation agent trained via pure reinforcement learning (GRPO / REINFORCE) with local 4-cell observations.


## Step 1: Mount Google Drive & Configure Workspace


In [ ]:
from google.colab import drive
import os, sys

try:
    drive.mount('/content/drive')
    DRIVE_WORKSPACE = '/content/drive/MyDrive/maze_workspace'
    os.makedirs(f'{DRIVE_WORKSPACE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_WORKSPACE}/runs', exist_ok=True)
    print(f'✓ Google Drive workspace ready: {DRIVE_WORKSPACE}')
except Exception as e:
    print(f'Drive 挂载提示: {e}')
    DRIVE_WORKSPACE = None

## Step 2: Environment Setup & Codebase Cloning


In [ ]:
!pip install -q torch openpyxl huggingface_hub matplotlib pandas

%cd /content
if not os.path.exists('/content/maze-transformer'):
    print('Cloning repository from Hugging Face...')
    !git clone https://huggingface.co/Hana-ame/maze-transformer /content/maze-transformer

%cd /content/maze-transformer
if '/content/maze-transformer' not in sys.path:
    sys.path.insert(0, '/content/maze-transformer')
print('✓ 迷宫导航环境与 Python 路径就绪')

## Step 3: Download Pretrained Weights from Hugging Face (.pt)


In [ ]:
import os, torch
from huggingface_hub import hf_hub_download

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)

REPO_ID = 'Hana-ame/maze-transformer'
CKPT_NAME = 'maze_grpo_final.pt'
os.makedirs('checkpoints', exist_ok=True)
local_path = f'checkpoints/{CKPT_NAME}'

print(f'Downloading primary maze model from Hugging Face: {CKPT_NAME} ...')
try:
    hf_hub_download(repo_id=REPO_ID, filename=f'checkpoints/{CKPT_NAME}', local_dir='.', token=hf_token)
    print(f'✓ 迷宫权重下载完成: {local_path} ({os.path.getsize(local_path)/1024:.1f} KB)')
    if DRIVE_WORKSPACE:
        !cp {local_path} {DRIVE_WORKSPACE}/checkpoints/{CKPT_NAME}
except Exception as e:
    print(f'下载提示: {e}，将直接进行从头 RL 训练。')

## Step 4: Generate Training Configuration (`maze_config.json`)


In [ ]:
import json

maze_cfg = {
    'layers': 2,                  # Optimal 2-layer Transformer
    'd': 64,                      # Hidden channel width 64
    'heads': 4,                   # 4 头精确对应上下左右 4 格视场
    'steps': 120,                 # 训练步数 (纯RL 120步收敛)
    'batch_size': 6,              # 轨迹批量
    'lr': 3e-4,                   # 强化学习最佳 LR
    'min_size': 5,                # Minimum maze size
    'max_size': 9,                # Maximum maze size
    'single': True,               # 单轨迹训练
    'datasource': {
        'type': 'random_perfect_maze',
        'observation': 'forced_obs_4cell',
        'reward': 'sparse_goal_reach'
    }
}

with open('maze_config.json', 'w', encoding='utf-8') as f:
    json.dump(maze_cfg, f, indent=2)

print('✓ 迷宫配置文件已生成 maze_config.json:')
print(json.dumps(maze_cfg, indent=2))

## Step 5: Launch Pure RL Training (`train.py --config maze_config.json`)


In [ ]:
# Run pure RL training and monitor solvability and reach rate progression
!python -m maze_transformer.train --config maze_config.json

## Step 6: Multi-Scale Maze Navigation Benchmark (5x5 - 9x9)


In [ ]:
# Run benchmark evaluating reach rates across maze dimensions
!python -m maze_transformer.bench || true

## Step 7: Dynamic INT8 Quantization Benchmark


In [ ]:
# Verify policy stability under dynamic INT8 quantization
!python -m maze_transformer.quantize --checkpoint checkpoints/maze_grpo_final.pt

## Step 8: Archive All Artifacts to Google Drive


In [ ]:
if DRIVE_WORKSPACE:
    !cp -ru runs/ {DRIVE_WORKSPACE}/runs/ || true
    !cp -ru checkpoints/ {DRIVE_WORKSPACE}/checkpoints/ || true
    print(f'✓ Maze models and logs archived to Google Drive: {DRIVE_WORKSPACE}')